In [ ]:
import duckdb
# path = "/home/bien/Documents/Development/RCS/robot-control-stack/utn_boxpnp_gripper"
path = "/home/bien/Documents/Development/RCS/datasets/dataset_parquet/utn_usbc_insertion_with_absolute_action_finger"
duckdb.sql(f"describe select *, from read_parquet('{path}')"), duckdb.sql(f"select count(*) as n_frames from read_parquet('{path}')")

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from matplotlib import pyplot as plt

episode = 13
uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}')"
).fetchnumpy()
episode = 0
n_frames = 400
frame_stride = 3
uuid1 = uuids["uuid"][episode]

df = duckdb.sql(f"""
    SELECT
        step,
        obs.right.tquat AS obs_tquat,
        obs.right.gripper AS obs_gripper,
        action.right.gripper AS action_gripper,
        obs.right.fingers.left_robot_frame AS finger_left,
        obs.right.fingers.right_robot_frame AS finger_right,
        obs.right.fingers.left_wrist_frame AS finger_left_w,
        obs.right.fingers.right_wrist_frame AS finger_right_w
    FROM read_parquet('{path}')
    WHERE uuid = '{uuid1}'
    ORDER BY step
""").df()

def unnest_fingers(finger):
    finger_unnested = []
    for i in range(finger.shape[0]):
        r0 = finger[i][0]
        r1 = finger[i][1]
        r2 = finger[i][2]
        r3 = finger[i][3]
        _m = np.stack([r0, r1, r2, r3])
        finger_unnested.append(_m)
    return np.stack(finger_unnested)
# Unnest fingers
left_finger = unnest_fingers(np.stack(df["finger_left"][1:].to_numpy()))
right_finger = unnest_fingers(np.stack(df["finger_right"][1:].to_numpy()))
left_finger_w = unnest_fingers(np.stack(df["finger_left_w"][1:].to_numpy()))
right_finger_w = unnest_fingers(np.stack(df["finger_right_w"][1:].to_numpy()))

left_finger_xyz = left_finger[:, :, 3]
right_finger_xyz = right_finger[:, :, 3]
left_finger_w_xyz = left_finger_w[:, :, 3]
right_finger_w_xyz = right_finger_w[:, :, 3]

# Expand quaternion columns into x/y/z/w components
obs_tquat = np.stack(df["obs_tquat"][1:].to_numpy())
obs_xyz = obs_tquat[:, :3]
steps = df["step"][1:]
print(left_finger_xyz.shape, right_finger_xyz.shape, obs_xyz.shape, steps.shape)

t_names = ['x','y','z']
fig, ax = plt.subplots(2, 3, figsize=(18, 8), sharex=True)
for i, name in enumerate(t_names):
    ax[0][i].plot(steps, obs_xyz[:, i], label="obs", linewidth=2)
    ax[0][i].plot(steps, right_finger_xyz[:, i], label="finger_right_0", linewidth=2, linestyle="--")
    ax[0][i].plot(steps, left_finger_xyz[:, i], label="finger_left_0", linewidth=2, linestyle="--")
    ax[0][i].set_title(f"{name}")
    ax[0][i].set_xlabel("step")
    ax[0][i].grid(True, alpha=0.3)
    ax[0][i].legend()
for i, name in enumerate(t_names):
    ax[1][i].plot(steps, right_finger_w_xyz[:, i], label="finger_right_w", linewidth=2, linestyle="--")
    ax[1][i].plot(steps, left_finger_w_xyz[:, i], label="finger_left_w", linewidth=2, linestyle="--")
    ax[1][i].set_title(f"{name} (wrist frame)")
    ax[1][i].set_xlabel("step")
    ax[1][i].grid(True, alpha=0.3)
    ax[1][i].legend()

plt.tight_layout()
plt.show()